## Topic 3: Text Loading

### Concept, Intuition, Why It Exists

- Targets: `fd_keyword_groups.txt`, `fd_negation_phrases.txt` — small, structured **reference** files, not prose documents.
- The defining decision: **load the whole file as ONE Document, never split it.** A keyword group only means something as a complete list — chunking it would tear related keywords apart and actively *hurt* retrieval, not help it. This is the "don't split this" exception to the usual rule.

### Internal Working

1. Open the file with **explicit UTF-8 encoding** — mixed-language terms will corrupt silently on a non-UTF-8 default.
2. Read the entire file content in one call.
3. Wrap it in exactly one `make_document()` call, with `metadata={"source": file_path}`.
4. A lazy generator yields that single Document; an eager version simply materializes the generator into a list.

### Real-World Issues, Edge Cases, Debugging

- **Encoding errors**: a file saved in `cp1252`/`latin-1` raises `UnicodeDecodeError` or — worse — silently produces garbled-but-valid-looking text if decoded with the wrong codec. Always pin `encoding="utf-8"` explicitly.
- **Empty file**: returns a Document with empty content, no error. Silently degrades retrieval — worth an explicit length check before accepting the Document.
- **Very large text files**: this loader assumes "small enough to read fully into memory" is safe — true for keyword lists, false if someone later points it at a huge log file.

### Design Decisions & Trade-offs

- Whole-file-as-one-Document vs. line-by-line: line-by-line would let you retrieve individual keyword groups, but these files are small enough that the entire file in one retrieval result is *more* useful than multiple stitched hits.
- This is a genuine granularity trade-off repeated throughout ingestion — the right granularity is "the smallest unit that's still semantically complete on its own," and that unit differs per source type.

### Alternatives & When To Use Each

- Whole file as one Document — small structured reference files where splitting destroys meaning.
- Line-by-line Documents — log files, line-delimited records where each line stands alone.
- Section-based splitting (by header markers) — larger structured text files with internal sections worth retrieving independently.

### Common Mistakes & Production Failures

- Reflexively running every source through the same chunker "for consistency," including small reference files that should never be split.
- Forgetting `encoding="utf-8"` — works fine locally, breaks in CI/production running a different default locale.

### Lead-Level Interview Questions

**Q: Why does this loader explicitly avoid chunking, when chunking is applied almost everywhere else?**
A: Chunking exists to keep retrieval units small and semantically coherent. For these files, the whole file already is one coherent unit — a keyword group is meaningless torn in half. Chunking here would optimize for chunk size at the expense of the actual goal, semantic completeness.

**Q: What's the actual failure mode of forgetting file encoding, and why might it pass review but fail in production?**
A: Locally, your editor/OS default likely matches the file's actual encoding, so it "just works." In production — a different container or locale — the default can differ, causing either a hard crash or silent corruption. Classic "works on my machine" bug, which is why explicit `encoding="utf-8"` is non-negotiable.

### Hidden Concepts Worth Knowing

- **BOM (Byte Order Mark)**: a few invisible bytes some editors prepend to UTF-8 files. `encoding="utf-8"` includes the BOM as a literal character; `encoding="utf-8-sig"` strips it.
- **Generators vs. lists**: trivial for a single file, but this establishes the convention every other loader follows — which matters a lot at scale.

### Revision Summary

> Text loading reads small, structured reference files as exactly ONE Document each — never split, because splitting would destroy the only thing that makes them useful. Always open with explicit `encoding="utf-8"`. This is the exception that proves chunking is a deliberate choice, not a default reflex.

In [2]:
"""
text_loader.py
---------------
Loads small structured reference .txt files (fd_keyword_groups.txt,
fd_negation_phrases.txt) as ONE Document per file.

Design choice: do NOT split these into multiple Documents. A keyword group
or a negation list only makes sense as a complete block -- chunking it would
tear related keywords apart and actively hurt retrieval quality.
"""

from document import make_document


def lazy_load_text(file_path: str):
    """Generator version -- reads the whole file once, yields exactly one Document."""
    with open(file_path, encoding="utf-8") as f:
        yield make_document(page_content=f.read(), metadata={"source": file_path})


def load_text(file_path: str) -> list:
    """Eager version -- same as lazy_load_text but returns a list, not a generator."""
    return list(lazy_load_text(file_path))


if __name__ == "__main__":
    docs = load_text("../data/fd_keyword_groups.txt")
    print(f"Loaded {len(docs)} document (whole file as one chunk)")
    print(f"  metadata     : {docs[0]['metadata']}")
    print(f"  content[:80] : {docs[0]['page_content'][:80]!r}")

Loaded 1 document (whole file as one chunk)
  metadata     : {'source': '../data/fd_keyword_groups.txt'}
  content[:80] : '# FD_provider — FD-signal keyword groups ONLY.\n# 4 groups, all positive FD-domai'
